# Laboratorio: SVM de clasificación binaria (supervisada)

Antes del **One-Class SVM** para anomalías, vemos la SVM **clásica con etiquetas** $y_i \in \{-1,+1\}$:

- busca un hiperplano (o frontera con kernel) que **maximiza el margen** entre clases,
- predicción: $\operatorname{sign}(w^{\top} x + b)$ o vía kernel y support vectors.

**Ejemplo A — Iris** (setosa vs versicolor, 2 medidas): kernel **lineal**.  
**Ejemplo B — Lunas** (`make_moons`): necesita kernel **RBF**.


## 0. Dependencias


In [ ]:
# Sin instalaciones extra.


## 1. Imports y gráfico de frontera


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris, make_moons
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

RANDOM_STATE = 42
plt.style.use('default')

def plot_decision_boundary_2d(model, X, y, title, ax=None):
    """Frontera, margenes y support vectors en 2D."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 5.5))
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300),
    )
    zz = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=[zz.min(), 0, zz.max()], colors=["#fee5d9", "#e5f5e0"], alpha=0.85)
    ax.contour(xx, yy, zz, levels=[0], colors="tab:red", linewidths=2, label="frontera f=0")
    ax.contour(xx, yy, zz, levels=[-1, 1], colors="tab:blue", linestyles="--", linewidths=1, alpha=0.7)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=40)
    sv = model.support_vectors_
    ax.scatter(
        sv[:, 0], sv[:, 1], s=140, facecolors="none", edgecolors="lime", linewidths=2,
        label=f"support vectors ({len(sv)})",
    )
    ax.set_title(title)
    ax.set_xlabel("feature 1")
    ax.set_ylabel("feature 2")
    ax.legend(loc="best")
    ax.grid(True, alpha=0.25)
    return ax


## 2. Iris — datos y SVM lineal

Solo **sepal length** y **sepal width** para poder dibujar en el plano.


In [ ]:
iris = load_iris()
mask = iris.target < 2
X_iris = iris.data[mask, :2]
y_iris_signed = np.where(iris.target[mask] == 0, -1, 1)
names = iris.target_names[:2]

X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris_signed, test_size=0.25, random_state=RANDOM_STATE, stratify=y_iris_signed
)
print('Clases:', list(names))
print('Train', X_train.shape, '| Test', X_test.shape)


In [ ]:
svm_linear = SVC(kernel='linear', C=1.0)
svm_linear.fit(X_train, y_train)

print('Accuracy train:', accuracy_score(y_train, svm_linear.predict(X_train)))
print('Accuracy test :', accuracy_score(y_test, svm_linear.predict(X_test)))
print()
print(classification_report(y_test, svm_linear.predict(X_test), target_names=names))
print('w =', svm_linear.coef_)
print('b =', svm_linear.intercept_)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_decision_boundary_2d(svm_linear, X_train, y_train, 'Iris (train) — SVM lineal', ax=axes[0])
plot_decision_boundary_2d(svm_linear, X_test, y_test, 'Iris (test)', ax=axes[1])
plt.tight_layout()
plt.show()


**Lectura del grafico**

- **Rojo solido:** frontera $f(x)=0$.
- **Azul punteado:** bandas de margen $f(x)=\pm 1$.
- **Verde:** support vectors — solo esos puntos fijan la solucion.


## 3. Efecto del parámetro `C` (margen suave)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, C in zip(axes, [0.1, 1.0, 100.0]):
    m = SVC(kernel='linear', C=C).fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    plot_decision_boundary_2d(
        m, X_train, y_train, f'C={C} | acc test={acc:.2f} | SV={len(m.support_vectors_)}', ax=ax
    )
fig.suptitle('Margen suave: trade-off C', y=1.02)
plt.tight_layout()
plt.show()


## 4. Lunas — lineal vs RBF

Datos **no** linealmente separables: la recta falla; el **RBF** curva la frontera (misma familia de kernel que en One-Class SVM).


In [ ]:
X_moon, y_moon = make_moons(n_samples=300, noise=0.18, random_state=RANDOM_STATE)
y_m_signed = np.where(y_moon == 0, -1, 1)

Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(
    X_moon, y_m_signed, test_size=0.25, random_state=RANDOM_STATE, stratify=y_m_signed
)

moon_lin = SVC(kernel='linear', C=1.0).fit(Xm_tr, ym_tr)
moon_rbf = SVC(kernel='rbf', gamma='scale', C=1.0).fit(Xm_tr, ym_tr)

print('Lunas — acc test | lineal:', accuracy_score(ym_te, moon_lin.predict(Xm_te)))
print('Lunas — acc test | RBF  :', accuracy_score(ym_te, moon_rbf.predict(Xm_te)))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_decision_boundary_2d(moon_lin, Xm_tr, ym_tr, 'Lunas — lineal', ax=axes[0])
plot_decision_boundary_2d(moon_rbf, Xm_tr, ym_tr, 'Lunas — RBF', ax=axes[1])
plt.tight_layout()
plt.show()


## 5. Forma dual (puente con la teoria)

$$
f(x) = \sum_{i \in \text{SV}} \alpha_i y_i\, k(x_i, x) + b
$$

- Lineal: $k(x,z)=x^{\top} z$
- RBF: $k(x,z)=\exp(-\gamma \|x-z\|^2)$


In [ ]:
print('Iris lineal — support vectors por clase:', svm_linear.n_support_)
print('Lunas RBF — support vectors por clase:', moon_rbf.n_support_)


## 6. Supervisada vs One-Class (KC1)

| | SVM binaria (este notebook) | One-Class SVM |
|--|------------------------------|---------------|
| Etiquetas | Sí ($\pm 1$) | No (solo “normal”) |
| Objetivo | Separar dos clases | Envolver una clase |
| Evaluación | Accuracy | Rareza / defectos |

**Ejercicios:** variar `C` en Iris; `gamma` en lunas; contar SV y explicar un punto mal clasificado.
